# 🐍 Notebook 08: Object-Oriented Programming, Exceptions & File I/O
**Trainer: Sundeep Hiremat · TASK Nalgonda Center**  
**Course:** Cloud Software Developer (AWS DVA-C02 Aligned)

In this notebook, we complete the core Python foundation:
1. **Object-Oriented Programming (OOP):** Modeling cloud resources with Classes, Inheritance, and Encapsulation.
2. **Exception Handling:** Building bulletproof code with `try / except / else / finally`.
3. **File Handling & JSON:** Managing local configuration files and AWS event payloads.


## 1. Object-Oriented Programming (OOP) for Cloud
We model cloud assets (EC2, S3, RDS) as objects with states and behaviors.


In [ ]:
# 1. Base Class (Parent)
class CloudResource:
    """Base class representing any AWS cloud resource."""
    cloud_provider = "AWS"  # Class attribute
    
    def __init__(self, resource_id: str, region: str = "ap-south-1"):
        self.resource_id = resource_id  # Public attribute
        self.region = region
        self._state = "PENDING"         # Protected attribute
        self.__secret_arn = f"arn:aws:secret:{resource_id}" # Private attribute
        
    def start(self):
        self._state = "RUNNING"
        print(f"🟢 Resource {self.resource_id} is now RUNNING in {self.region}.")
        
    def stop(self):
        self._state = "STOPPED"
        print(f"🔴 Resource {self.resource_id} is now STOPPED.")
        
    def get_status(self):
        return self._state

# 2. Subclass / Inheritance (Child)
class EC2Instance(CloudResource):
    """Inherits from CloudResource and adds compute-specific features."""
    def __init__(self, resource_id: str, instance_type: str = "t3.micro", region: str = "ap-south-1"):
        super().__init__(resource_id, region)
        self.instance_type = instance_type
        
    # Method Overriding (Polymorphism)
    def start(self):
        super().start()
        print(f"  ⚡ Booting virtual CPU on {self.instance_type} compute core...")

# Create and test
server = EC2Instance("i-04a29c7811", instance_type="t3.medium")
server.start()
print("State:", server.get_status())
server.stop()


---
## 2. Robust Exception Handling (`try / except / else / finally`)
Cloud operations must handle missing resources, API timeouts, and quota limits gracefully.


In [ ]:
# Custom Domain Exception
class BudgetExceededError(Exception):
    """Raised when cloud spending exceeds authorized monthly cap."""
    pass

def provision_fleet(server_count: int, unit_cost: float, budget_limit: float):
    try:
        if server_count <= 0:
            raise ValueError("Server count must be at least 1!")
            
        total_cost = server_count * unit_cost
        if total_cost > budget_limit:
            raise BudgetExceededError(
                f"Cannot launch {server_count} instances! Total: ${total_cost}, Cap: ${budget_limit}"
            )
            
        print(f"🎉 Successfully provisioned {server_count} instances for ${total_cost}.")
        return True

    except ValueError as ve:
        print(f"❌ Input Validation Error: {ve}")
        return False
    except BudgetExceededError as be:
        print(f"🛑 Cost Shield Triggered: {be}")
        return False
    except Exception as e:
        print(f"💥 Unexpected Error: {e}")
        return False
    else:
        print("✅ Execution completed without any errors.")
    finally:
        print("🔒 [Audit System] Provisioning attempt logged to CloudTrail.")

# Test scenarios
provision_fleet(server_count=10, unit_cost=50.0, budget_limit=300.0) # Triggers budget error
print("-" * 50)
provision_fleet(server_count=3, unit_cost=50.0, budget_limit=300.0)  # Succeeds


---
## 3. File I/O & JSON Serialization (Cloud Configs)
In AWS, configuration files, IAM policies, and Lambda event triggers are all JSON.


In [ ]:
import json

# Python Dictionary representing cloud architecture
deployment_manifest = {
    "Project": "Cloud Portfolio",
    "Environment": "Production",
    "Center": "TASK Nalgonda",
    "Trainer": "Sundeep Hiremat",
    "EC2_Instances": [
        {"id": "i-web-01", "type": "t3.micro", "public_ip": "13.233.45.10"},
        {"id": "i-web-02", "type": "t3.micro", "public_ip": "13.233.45.11"}
    ],
    "SecurityGroup": {
        "Name": "web-sg",
        "InboundPorts": [80, 443, 22]
    }
}

# 1. Writing to JSON file using context manager ('with')
filename = "cloud_manifest.json"
with open(filename, "w", encoding="utf-8") as f:
    json.dump(deployment_manifest, f, indent=4)
print(f"✅ Created '{filename}' on disk.")

# 2. Reading back and deserializing
with open(filename, "r", encoding="utf-8") as f:
    loaded_config = json.load(f)

print("\n📂 Loaded from JSON:")
print(f"Project: {loaded_config['Project']} by {loaded_config['Trainer']}")
print(f"Open Inbound Ports: {loaded_config['SecurityGroup']['InboundPorts']}")


---
## 4. File I/O Mastery: Access Modes, Pointer Navigation (`tell` & `seek`) and Safe Handling

### 📌 The Basics of File I/O (3-Step Lifecycle)
1. **Open**: `open()` connects to the file on disk and returns a file object / pointer.
2. **Operate**: Read (`.read()`, `.readline()`) or write (`.write()`, `.writelines()`).
3. **Close**: `close()` releases the operating system handle back to Windows/Linux. The `with` statement ensures this automatically!

### 📊 File Access Modes Matrix
| Mode | Action | File Must Exist? | Truncates (Overwrites) Data? | Pointer Position |
| :--- | :--- | :--- | :--- | :--- |
| `'r'` | Read only (Default) | **Yes** (throws `FileNotFoundError`) | No | Beginning of file |
| `'w'` | Write only | No (creates new file if missing) | **Yes** (wipes entire file!) | Beginning of file |
| `'a'` | Append only | No (creates if missing) | No (preserves existing data) | **End** of file |
| `'r+'` | Read + Write | **Yes** | No | Beginning of file |
| `'w+'` | Read + Write | No (creates if missing) | **Yes** (wipes clean first) | Beginning of file |
| `'a+'` | Read + Append | No (creates if missing) | No | End of file |
| `'b'` | Binary Mode (e.g., `'rb'`, `'wb'`) | Needed for non-text files (images, PDFs, ZIPs) | — | — |

### 🎯 The Cassette Tape Analogy: `tell()` vs `seek()`
- **`file.tell()`**: Like checking the counter on a cassette tape. It returns the current byte position where reading or writing will occur.
- **`file.seek(offset, whence)`**: Like fast-forwarding or rewinding the cassette tape to a specific timestamp.
  - `whence = 0`: Offset relative to the beginning of the file (Default).
  - `whence = 1`: Offset relative to the current file pointer position.
  - `whence = 2`: Offset relative to the end of the file.


In [ ]:
# File Pointer Manipulation with tell(), seek() and Exception Handling
sample_log = 'task_access_log.txt'

# Step 1: Write initial sample text
with open(sample_log, 'w', encoding='utf-8') as f:
    f.write('ABCDEFGHIJKLMNOPQRSTUVWXYZ')
print(f"Initial file written: {sample_log}")

# Step 2: Navigate using tell() and seek() inside try/except/finally
f_obj = None
try:
    f_obj = open(sample_log, 'r+', encoding='utf-8')
    print(f"1. Starting position (tell): {f_obj.tell()}")  # 0
    
    chunk = f_obj.read(5)
    print(f"2. Read 5 characters: '{chunk}'")
    print(f"3. Position after reading 5 chars: {f_obj.tell()}")  # 5
    
    # Rewind pointer back to index 2 (Letter 'C')
    f_obj.seek(2, 0)
    print(f"4. Position after seek(2): {f_obj.tell()}")
    next_chars = f_obj.read(4)
    print(f"5. Read 4 chars from offset 2: '{next_chars}'")  # 'CDEF'
    
    # Jump to end and append safely
    f_obj.seek(0, 2)  # Seek to end
    print(f"6. End of file position: {f_obj.tell()}")
    f_obj.write('-COMPLETED')

except FileNotFoundError as err:
    print(f"[ERROR] Target log file missing: {err}")
except PermissionError:
    print('[ERROR] Permission denied by operating system.')
except Exception as err:
    print(f"[ERROR] Unexpected error: {err}")
else:
    print('✅ [ELSE] File pointer operations executed without any errors!')
finally:
    if f_obj and not f_obj.closed:
        f_obj.close()
        print('🔒 [FINALLY] File handle explicitly released back to OS.')

# Verify final content
with open(sample_log, 'r', encoding='utf-8') as f:
    print('Final File Content:', f.read())

# Clean up demo file
import os
if os.path.exists(sample_log):
    os.remove(sample_log)


---
## 5. Python Standard Libraries: Why & How to Use Them

### Why do we use Standard Libraries?
> **Analogy for Students:** Python comes 'batteries included'. You do not need to build network protocols, regex pattern engines, or file-copy routines from scratch. The standard library provides battle-tested, high-performance tools built into every Python installation without running `pip install`.

### Key Standard Libraries Covered:
1. **`os` & `sys`**: Interacting with host filesystem paths, directories, and runtime arguments.
2. **`shutil`**: High-level file and directory actions (copying, recursive deleting `rmtree`).
3. **`glob`**: Wildcard pattern search (e.g., `*.py` or `**/*.log` across all subfolders).
4. **`re`**: Regular expressions for validating formats (emails, IPs, phones) and parsing server logs.
5. **`urllib.request`**: Built-in HTTP client for fetching web API endpoints.
6. **`smtplib`**: Simple Mail Transfer Protocol client for automated email alerts.


In [ ]:
import os
import shutil
import glob
import re
import urllib.request
import smtplib
from email.mime.text import MIMEText

print('=' * 60)
print('1. OS, SHUTIL & GLOB (Filesystem Operations)')
print('=' * 60)
# OS: Current working directory and path joining
current_dir = os.getcwd()
safe_path = os.path.join(current_dir, 'sample_folder')
print(f'OS Safe Path: {safe_path}')

# Glob: Pattern matching (find all ipynb notebooks)
notebooks = glob.glob('*.ipynb')
print(f'Glob found notebooks: {notebooks[:3]}')

print('\n' + '=' * 60)
print('2. REGEX (re Module: Search, Findall, Sub)')
print('=' * 60)
server_log = "2026-11-22 14:30:10 WARN [IP: 192.168.1.105] Failed login for user 'admin_nalgonda'"

# re.search: Find first match (IP address pattern)
ip_match = re.search(r'\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}', server_log)
if ip_match:
    print(f'Extracted IP Address: {ip_match.group()}')

# re.sub: Mask sensitive IP for privacy
clean_log = re.sub(r'\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}', '[REDACTED_IP]', server_log)
print(f'Masked Log Entry: {clean_log}')

print('\n' + '=' * 60)
print('3. URLLIB (Web Requests) & SMTPLIB (Email Alerts)')
print('=' * 60)
# Fetching API response safely using urllib
api_url = 'https://httpbin.org/get'
try:
    with urllib.request.urlopen(api_url, timeout=3) as resp:
        print(f'HTTP Status: {resp.status} - Content Type: {resp.getheader("Content-Type")}')
except Exception as net_err:
    print(f'Note: Offline or network restricted ({net_err})')

# smtplib email template construction
email_msg = MIMEText('Server alert: CPU utilization exceeded 90%.')
email_msg['Subject'] = 'TASK Cloud Notification'
email_msg['From'] = 'monitoring@nalgonda-cloud.edu'
email_msg['To'] = 'trainer@nalgonda-cloud.edu'
print('SMTP Message Prepared:\n', email_msg.as_string().strip())
